# 1 - Téléchargement et préparation Food-101 (10 classes)

Ce notebook télécharge **Food-101** depuis Kaggle (`dansbecker/food-101`),
sélectionne **10 classes visuellement distinctes** et prépare la structure multi-fidélité :

- `test/` : 10 % des images par classe (100 images × 10 classes = 1 000 images)
- `train/HF/` : 10 % du train (90 images × 10 classes = 900 images, Haute Fidélité)
- `train/BF/` : 90 % du train (810 images × 10 classes = 8 100 images, Basse Fidélité)

**Classes sélectionnées (10)** : pizza, hamburger, hot_dog, sushi, ice_cream,
french_fries, ramen, steak, waffles, fried_rice

**Prérequis Colab** : dans l'onglet `Secrets`, ajouter `KAGGLE_USERNAME` et `KAGGLE_KEY`
(disponibles sur kaggle.com → votre compte → API → Create New Token).

In [ ]:
import sys, os, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

# Identifiants Kaggle : secrets Colab, sinon ~/.kaggle/kaggle.json (serveur/local).
if env_config.in_colab():
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print('Credentials Kaggle configures depuis les secrets Colab.')
    except Exception as e:
        print(f'Secrets Colab non disponibles : {e}')

# Telechargement via kagglehub (Colab ET serveur ; ~/.kaggle/kaggle.json requis sur serveur)
os.system('pip install -q kagglehub')
import kagglehub
FOOD101_RAW_PATH = kagglehub.dataset_download('dansbecker/food-101')
print('FOOD101_RAW_PATH:', FOOD101_RAW_PATH)

In [ ]:
import sys, os, importlib, shutil, random
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

# 10 classes visuellement distinctes (textures, formes, couleurs tres differentes)
SELECTED_CLASSES = [
    'pizza', 'hamburger', 'hot_dog', 'sushi', 'ice_cream',
    'french_fries', 'ramen', 'steak', 'waffles', 'fried_rice'
]

def find_images_dir(base):
    """Trouve le dossier 'images/' contenant les sous-dossiers par classe."""
    for candidate in [
        os.path.join(base, 'food-101', 'images'),
        os.path.join(base, 'images'),
        base,
    ]:
        if os.path.isdir(candidate) and any(
                os.path.isdir(os.path.join(candidate, d)) for d in os.listdir(candidate)):
            return candidate
    raise FileNotFoundError(f'Impossible de trouver le dossier images/ dans {base}')

images_dir = find_images_dir(FOOD101_RAW_PATH)
print('Dossier images:', images_dir)
available = sorted(os.listdir(images_dir))
missing = [c for c in SELECTED_CLASSES if c not in available]
if missing:
    print('ATTENTION - classes introuvables:', missing)
else:
    print('Toutes les classes selectionnees sont presentes.')

out_dir = env_config.data_dir('Food101')
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
for d in [f'{out_dir}/test', f'{out_dir}/train/HF', f'{out_dir}/train/BF']:
    os.makedirs(d, exist_ok=True)

for cls in SELECTED_CLASSES:
    cls_dir = os.path.join(images_dir, cls)
    images = [f for f in os.listdir(cls_dir)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.seed(42)
    random.shuffle(images)
    n_test = int(len(images) * 0.10)
    test_imgs  = images[:n_test]
    train_imgs = images[n_test:]
    n_hf = int(len(train_imgs) * 0.10)
    hf_imgs = train_imgs[:n_hf]
    bf_imgs = train_imgs[n_hf:]
    for split, split_imgs in [('test', test_imgs), ('train/HF', hf_imgs), ('train/BF', bf_imgs)]:
        dest = os.path.join(out_dir, split, cls)
        os.makedirs(dest, exist_ok=True)
        for n in split_imgs:
            shutil.copy2(os.path.join(cls_dir, n), os.path.join(dest, n))

for split_name in ['test', 'train/HF', 'train/BF']:
    total = sum(len(os.listdir(os.path.join(out_dir, split_name, c)))
                for c in SELECTED_CLASSES
                if os.path.isdir(os.path.join(out_dir, split_name, c)))
    print(f'{split_name:12s}: {total} images')
print('Termine ! Food101 multi-fidelite dans:', out_dir)

In [ ]:
import sys, os, importlib, shutil
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

# Sur Colab : zipper + uploader sur le Drive. Sur serveur/local : rien a faire.
if env_config.in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = '/content/drive/MyDrive/UTBM_PF22/datasets/Food101'
    os.makedirs(drive_path, exist_ok=True)
    get_ipython().system('cd /content && zip -0 -r -q food101_multifidelity.zip processed_multifidelity/')
    shutil.copy2('/content/food101_multifidelity.zip', f'{drive_path}/dataset_multifidelity.zip')
    print('Zip Food101 uploade sur le Drive.')
else:
    print('Serveur/local : pas de zip necessaire (donnees deja sur le disque).')